# LAB | Agents that speak MCP fluently

## Step 1: Setup and Installation

In [ ]:
%pip install langchain langchain-openai langchain-mcp-adapters mcp python-dotenv

In [1]:
import os
from dotenv import load_dotenv

load_dotenv()

for key in ["OPENAI_API_KEY", "PINECONE_API_KEY"]:
    if not os.getenv(key):
        raise ValueError(f"{key} not found. Add it to your .env file.")
print("API keys loaded")

API keys loaded


## Step 2: Connect to MCP Server

In [ ]:
from pathlib import Path
from langchain_mcp_adapters.client import MultiServerMCPClient

DOCS_DIR = Path("sample_docs").resolve()

# The official filesystem server runs via npx over stdio and can only touch the directories passed as args.
client = MultiServerMCPClient(
    {
        "filesystem": {
            "transport": "stdio",
            "command": "npx",
            "args": ["-y", "@modelcontextprotocol/server-filesystem", str(DOCS_DIR)],
        }
    }
)

In [5]:
from mcp.shared.exceptions import McpError

# Test the connection with a raw MCP session: list the server's tools and resources.
async with client.session("filesystem") as session:
    tools_result = await session.list_tools()
    print(f"{len(tools_result.tools)} tools:")
    for t in tools_result.tools:
        print(f"  - {t.name}: {t.description[:80]}")

    # Not every server implements resources; the filesystem server does not.
    try:
        resources_result = await session.list_resources()
        print(f"\n{len(resources_result.resources)} resources:", [str(r.uri) for r in resources_result.resources])
    except McpError as e:
        print(f"\nThis server exposes no resources ({e})")


14 tools:
  - read_file: Read the complete contents of a file as text. DEPRECATED: Use read_text_file ins
  - read_text_file: Read the complete contents of a file from the file system as text. Handles vario
  - read_media_file: Read a file and return it as a base64-encoded content block with its MIME type. 
  - read_multiple_files: Read the contents of multiple files simultaneously. This is more efficient than 
  - write_file: Create a new file or completely overwrite an existing file with new content. Use
  - edit_file: Make line-based edits to a text file. Each edit replaces exact line sequences wi
  - create_directory: Create a new directory or ensure a directory exists. Can create multiple nested 
  - list_directory: Get a detailed listing of all files and directories in a specified path. Results
  - list_directory_with_sizes: Get a detailed listing of all files and directories in a specified path, includi
  - directory_tree: Get a recursive tree view of files and directories as a 

## Step 3: Load MCP Tools into LangChain

In [6]:
# The instructions mention get_langchain_tools(); in langchain-mcp-adapters 0.3.x the method is get_tools().
tools = await client.get_tools()

print(f"Loaded {len(tools)} LangChain tools:")
for tool in tools:
    print(f"  - {tool.name} ({type(tool).__name__})")

Loaded 14 LangChain tools:
  - read_file (StructuredTool)
  - read_text_file (StructuredTool)
  - read_media_file (StructuredTool)
  - read_multiple_files (StructuredTool)
  - write_file (StructuredTool)
  - edit_file (StructuredTool)
  - create_directory (StructuredTool)
  - list_directory (StructuredTool)
  - list_directory_with_sizes (StructuredTool)
  - directory_tree (StructuredTool)
  - move_file (StructuredTool)
  - search_files (StructuredTool)
  - get_file_info (StructuredTool)
  - list_allowed_directories (StructuredTool)


In [7]:
# Verify the tools are accessible: inspect one tool's schema, then call it directly.
tools_by_name = {t.name: t for t in tools}

read_tool = tools_by_name["read_text_file"]
print("Args schema:", read_tool.args)

result = await read_tool.ainvoke({"path": str(DOCS_DIR / "mcp_overview.md")})
print("\nTool output:\n", result)

Args schema: {'path': {'type': 'string'}, 'tail': {'description': 'If provided, returns only the last N lines of the file', 'type': 'number'}, 'head': {'description': 'If provided, returns only the first N lines of the file', 'type': 'number'}}

Tool output:
 [{'type': 'text', 'text': '# MCP overview\n\nThe Model Context Protocol (MCP) is an open standard that lets AI applications connect to external tools and data.\n\n- **Tools** are functions a model can call, such as reading a file.\n- **Resources** are read-only data a server exposes for context.\n- **Prompts** are reusable templates a server offers to clients.\n\nAn MCP client talks to a server over a transport. Local servers usually use stdio; remote servers use HTTP.\n', 'id': 'lc_9c730b29-bb23-407b-be7b-e6f7c6fedbac'}]


## Step 4: Create Agent with MCP Tools

In [8]:
from langchain.agents import create_agent
from langchain_openai import ChatOpenAI

llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)

SYSTEM_PROMPT = f"""You are a document assistant with filesystem tools (via MCP).
You can only access this directory: {DOCS_DIR}
Always use the tools to read files before answering questions about their content.
Answer only from what the tools return. If the files do not contain the answer, say so.
Never write, edit, or move files unless explicitly asked."""

agent = create_agent(llm, tools, system_prompt=SYSTEM_PROMPT)

In [9]:
# Test with a query that can only be answered by calling an MCP tool.
query = "Using the files available to you, what are the three things an MCP server can offer a client?"
response = await agent.ainvoke({"messages": [{"role": "user", "content": query}]})

# Trace: every message in order (human -> tool call -> tool result -> final answer).
for msg in response["messages"]:
    if msg.type == "ai" and msg.tool_calls:
        for call in msg.tool_calls:
            print(f"[TOOL CALL] {call['name']}({call['args']})")
    elif msg.type == "tool":
        print(f"[TOOL RESULT] {msg.name}: {str(msg.content)[:200]}...")
    elif msg.type == "ai":
        print(f"\n[FINAL ANSWER]\n{msg.content}")

[TOOL CALL] search_files({'path': '/Users/ben-macbook-air/Files/Projects/Ironhack Bootcamp/course work/week-4 /lab-mcp-langchain/sample_docs', 'pattern': '*.txt'})
[TOOL RESULT] search_files: [{'type': 'text', 'text': 'No matches found', 'id': 'lc_eccb1348-a7b9-454d-8669-d6b234e2b11e'}]...
[TOOL CALL] search_files({'path': '/Users/ben-macbook-air/Files/Projects/Ironhack Bootcamp/course work/week-4 /lab-mcp-langchain/sample_docs', 'pattern': '*.*'})
[TOOL RESULT] search_files: [{'type': 'text', 'text': '/Users/ben-macbook-air/Files/Projects/Ironhack Bootcamp/course work/week-4 /lab-mcp-langchain/sample_docs/mcp_overview.md', 'id': 'lc_5028c7a4-f252-438e-baa4-d2572d004565'}]...
[TOOL CALL] read_text_file({'path': '/Users/ben-macbook-air/Files/Projects/Ironhack Bootcamp/course work/week-4 /lab-mcp-langchain/sample_docs/mcp_overview.md'})
[TOOL RESULT] read_text_file: [{'type': 'text', 'text': '# MCP overview\n\nThe Model Context Protocol (MCP) is an open standard that lets AI applications

## Step 5: Access MCP Resources

The filesystem server has no resources, so this step uses a second off-the-shelf server, `@modelcontextprotocol/server-everything` (the official reference server), which exposes documentation files as resources.

In [10]:
# Separate client for the resources server, so its 16 demo tools don't end up in the filesystem agent.
resource_client = MultiServerMCPClient(
    {
        "everything": {
            "transport": "stdio",
            "command": "npx",
            "args": ["-y", "@modelcontextprotocol/server-everything"],
        }
    }
)

# List resources (and resource templates) with the raw MCP session.
async with resource_client.session("everything") as session:
    resources = (await session.list_resources()).resources
    templates = (await session.list_resource_templates()).resourceTemplates

print(f"{len(resources)} static resources:")
for r in resources:
    print(f"  - {r.uri} ({r.mimeType})")
print("\nDynamic resource templates:", [t.uriTemplate for t in templates])

7 static resources:
  - demo://resource/static/document/architecture.md (text/markdown)
  - demo://resource/static/document/extension.md (text/markdown)
  - demo://resource/static/document/features.md (text/markdown)
  - demo://resource/static/document/how-it-works.md (text/markdown)
  - demo://resource/static/document/instructions.md (text/markdown)
  - demo://resource/static/document/startup.md (text/markdown)
  - demo://resource/static/document/structure.md (text/markdown)

Dynamic resource templates: ['demo://resource/dynamic/text/{resourceId}', 'demo://resource/dynamic/blob/{resourceId}']


In [11]:
# Read one resource. get_resources() returns LangChain Blob objects with the uri in metadata.
FEATURES_URI = "demo://resource/static/document/features.md"
blobs = await resource_client.get_resources("everything", uris=[FEATURES_URI])

features_doc = blobs[0].as_string()
print("URI:", blobs[0].metadata["uri"])
print(f"Length: {len(features_doc)} characters")
print(features_doc[:400])

URI: demo://resource/static/document/features.md
Length: 16839 characters
# Everything Server - Features

**[Architecture](architecture.md)
| [Project Structure](structure.md)
| [Startup Process](startup.md)
| Server Features
| [Extension Points](extension.md)
| [How It Works](how-it-works.md)**

## Protocol eras

The server serves two protocol eras from one process, over stdio and over Streamable HTTP. A client's first message picks its era:

- **2026-07-28** (`modern`


In [12]:
# Use the resource as agent context: put it in the prompt, no tool call needed.
question = "Which two protocol eras does the server support, and what does a client in the 2026-07-28 era send first?"

messages = [
    {"role": "system", "content": "Answer only from the document below. If it does not contain the answer, say so.\n\n"
                                   f"--- RESOURCE {FEATURES_URI} ---\n{features_doc}"},
    {"role": "user", "content": question},
]
answer = await llm.ainvoke(messages)

print("Question:", question)
print("\nAnswer:", answer.content)

Question: Which two protocol eras does the server support, and what does a client in the 2026-07-28 era send first?

Answer: The server supports two protocol eras: **2026-07-28** (`modern`) and **2025-era** (`legacy`). A client in the 2026-07-28 era sends its first message as `server/discover`, with no `initialize` and no session, and sends its protocol version, identity, and capabilities in each request's `_meta`.


## Step 6: Build a Complete MCP-Enabled Agent

In [13]:
# One client managing both servers: filesystem (tools) and everything (tools + resources).
multi_client = MultiServerMCPClient(
    {
        "filesystem": {
            "transport": "stdio",
            "command": "npx",
            "args": ["-y", "@modelcontextprotocol/server-filesystem", str(DOCS_DIR)],
        },
        "everything": {
            "transport": "stdio",
            "command": "npx",
            "args": ["-y", "@modelcontextprotocol/server-everything"],
        },
    }
)

# Tools from both servers, plus the features.md resource as background context.
all_tools = await multi_client.get_tools()
context_blobs = await multi_client.get_resources("everything", uris=[FEATURES_URI])
context_doc = context_blobs[0].as_string()

print(f"{len(all_tools)} tools loaded from {len(multi_client.connections)} servers")
print(f"Context resource: {context_blobs[0].metadata['uri']} ({len(context_doc)} chars)")

30 tools loaded from 2 servers
Context resource: demo://resource/static/document/features.md (16839 chars)


In [14]:
FULL_SYSTEM_PROMPT = f"""You are a research assistant connected to two MCP servers.

TOOLS
- Filesystem tools can read files in this directory only: {DOCS_DIR}
- Use filesystem tools to read files before answering questions about them.
- Other tools (echo, get-sum, ...) are demo utilities from the "everything" server.

BACKGROUND CONTEXT (the "features doc": an MCP resource already loaded below, NOT a file on disk;
never look for it with filesystem tools)
--- RESOURCE {FEATURES_URI} ---
{context_doc}

RULES
- Say which source each fact came from: a file you read with a tool, or the background resource.
- Answer only from those sources. If they do not contain the answer, say so.
- Never write, edit, or move files unless explicitly asked."""

full_agent = create_agent(llm, all_tools, system_prompt=FULL_SYSTEM_PROMPT)

In [15]:
async def ask(agent, query):
    """Run the agent and print the tool trace plus the final answer."""
    response = await agent.ainvoke({"messages": [{"role": "user", "content": query}]})
    for msg in response["messages"]:
        if msg.type == "ai" and msg.tool_calls:
            for call in msg.tool_calls:
                print(f"[TOOL CALL] {call['name']}({call['args']})")
        elif msg.type == "tool":
            print(f"[TOOL RESULT] {msg.name}: {str(msg.content)[:150]}...")
        elif msg.type == "ai":
            print(f"\n[FINAL ANSWER]\n{msg.content}")

# Needs both: a file read through a tool AND the resource context.
await ask(
    full_agent,
    "Read mcp_overview.md and tell me which transports it mentions for local and remote servers. "
    "Then tell me which transports the Everything server serves according to its features doc.",
)

[TOOL CALL] read_text_file({'path': '/Users/ben-macbook-air/Files/Projects/Ironhack Bootcamp/course work/week-4 /lab-mcp-langchain/sample_docs/mcp_overview.md'})
[TOOL RESULT] read_text_file: [{'type': 'text', 'text': '# MCP overview\n\nThe Model Context Protocol (MCP) is an open standard that lets AI applications connect to external tools ...
[TOOL CALL] read_text_file({'path': '/Users/ben-macbook-air/Files/Projects/Ironhack Bootcamp/course work/week-4 /lab-mcp-langchain/sample_docs/mcp_overview.md'})
[TOOL RESULT] read_text_file: [{'type': 'text', 'text': '# MCP overview\n\nThe Model Context Protocol (MCP) is an open standard that lets AI applications connect to external tools ...

[FINAL ANSWER]
The `mcp_overview.md` file mentions the following transports for local and remote servers:

- **Local servers** usually use **stdio**.
- **Remote servers** use **HTTP**.

According to the features document of the Everything server, it serves two protocol eras over the following transports:



In [16]:
# Cleanup. In langchain-mcp-adapters 0.3.x, client.get_tools() opens a fresh stdio session for each tool call
# and closes it afterwards, and client.session() closes on exit of its `async with` block. So there is no
# long-lived connection to disconnect and MultiServerMCPClient has no close() method.
print("Open sessions to close: none (sessions are per call)")

Open sessions to close: none (sessions are per call)
